# Структурированный вывод данных из заявок (ветка llm-improve-2)

Улучшение решения: вместо одного числа извлекаем из каждой заявки структурированный JSON с полями:

- `count_adults` — число взрослых;
- `count_children` — число детей;
- `start_date` — дата заезда;
- `nights` — число ночей;
- `price_per_day` — желаемая цена в сутки;
- `remarks` — особые пожелания (животные, детская кроватка, парковка и т.п.).

Используем системный и пользовательский промпты, пошаговые инструкции (Chain-of-Thought) и пример (few-shot), а результат парсим через `JsonOutputParser`. Точность считаем на ручной разметке 15 заявок.

In [ ]:
import os
import json
import pandas as pd
from dotenv import load_dotenv

from langchain_gigachat.chat_models import GigaChat
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

load_dotenv()
api_key = os.getenv("GIGA_KEY")
if not api_key:
    raise ValueError("Ключ GIGA_KEY не найден в .env")

llm = GigaChat(
    credentials=api_key,
    model="GigaChat-2",
    verify_ssl_certs=False,
    temperature=0.2,
    max_tokens=1000,
)

## 1. Схема JSON и промпт

`JsonOutputParser` сам подставляет инструкции по формату через `format_instructions`.

In [ ]:
json_schema = {
    "type": "object",
    "properties": {
        "count_adults": {"type": "integer", "description": "Число взрослых"},
        "count_children": {"type": "integer", "description": "Число детей"},
        "start_date": {"type": "string", "description": "Дата заезда (самая ранняя при диапазоне)"},
        "nights": {"type": "integer", "description": "Число ночей (максимум при диапазоне)"},
        "price_per_day": {"type": "integer", "description": "Желаемая цена в сутки (0/None, если не указано)"},
        "remarks": {"type": "string", "description": "Особые пожелания"},
    },
    "required": ["count_adults", "count_children", "start_date", "nights"],
}

parser = JsonOutputParser()

system_text = (
    "Ты — эксперт по анализу текстовых заявок на аренду жилья. "
    "Ты извлекаешь структурированные данные и возвращаешь строго JSON по заданной схеме."
)

human_template = """
Проанализируй заявку на аренду жилья и извлеки данные, следуя пошаговому алгоритму.
Заявка: {text}

ШАГ 1. Определи число взрослых (count_adults) и детей (count_children).
ШАГ 2. Определи дату заезда (start_date); при «плавающем» диапазоне — самую раннюю.
ШАГ 3. Определи число ночей (nights); при диапазоне — максимальное.
ШАГ 4. Определи желаемую цену в сутки (price_per_day); при диапазоне — максимум.
ШАГ 5. Выпиши особые пожелания (remarks): животные, детская кроватка, парковка и т.п.

Пример:
Заявка: "Ищем жильё с 22.07 по 31.07 на 2 взрослых и 1 ребёнка, с детской кроваткой, до 3000 за сутки"
Ответ: {{"count_adults": 2, "count_children": 1, "start_date": "22.07", "nights": 9, "price_per_day": 3000, "remarks": "детская кроватка"}}

{format_instructions}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_text),
    ("human", human_template),
]).partial(format_instructions=parser.get_format_instructions())

chain = prompt | llm | parser

## 2. Прогон по 15 заявкам варианта 19

In [ ]:
df = pd.read_csv("rental_19.csv", sep=";", encoding="utf-8")

predictions = []
for text in df["text"]:
    try:
        res = chain.invoke({"text": text})
    except Exception as e:  # noqa: BLE001
        res = {"error": str(e)}
    predictions.append(res)

for i, res in enumerate(predictions, 1):
    print(f"#{i}: {json.dumps(res, ensure_ascii=False)}")

## 3. Ручная разметка (эталон)

Для оценки точности вручную размечены 15 заявок. В ночном расчёте используем разницу дат (например, «с 1.09 по 8.09» → 7 ночей), дату берём самую раннюю из диапазона.

In [ ]:
refs = [
    {"count_adults": 2, "count_children": 0, "start_date": "16.07", "nights": 13, "price_per_day": None},
    {"count_adults": 2, "count_children": 4, "start_date": "29.05", "nights": 11, "price_per_day": None},
    {"count_adults": 2, "count_children": 2, "start_date": "15.06", "nights": 14, "price_per_day": None},
    {"count_adults": 2, "count_children": 0, "start_date": "01.09", "nights": 7, "price_per_day": None},
    {"count_adults": 2, "count_children": 0, "start_date": "07.09", "nights": 7, "price_per_day": None},
    {"count_adults": 2, "count_children": 1, "start_date": "17.08", "nights": 10, "price_per_day": None},
    {"count_adults": 3, "count_children": 0, "start_date": "18.08", "nights": 7, "price_per_day": None},
    {"count_adults": 4, "count_children": 0, "start_date": "19.08", "nights": 10, "price_per_day": None},
    {"count_adults": 2, "count_children": 2, "start_date": "11.08", "nights": 12, "price_per_day": None},
    {"count_adults": 3, "count_children": 2, "start_date": "30.07", "nights": 10, "price_per_day": None},
    {"count_adults": 2, "count_children": 2, "start_date": "04.07", "nights": 20, "price_per_day": None},
    {"count_adults": 1, "count_children": 0, "start_date": "10.07", "nights": 10, "price_per_day": None},
    {"count_adults": 2, "count_children": 0, "start_date": "22.07", "nights": 9, "price_per_day": None},
    {"count_adults": 2, "count_children": 1, "start_date": "25.07", "nights": 12, "price_per_day": None},
    {"count_adults": 3, "count_children": 0, "start_date": "08.08", "nights": 12, "price_per_day": None},
]

## 4. Точность по каждому полю и средняя

In [ ]:
def norm_date(v):
    if not v or not isinstance(v, str):
        return None
    return v.strip().replace("-", ".")[:5]


def norm_int(v):
    try:
        return int(v)
    except (TypeError, ValueError):
        return None


total = len(refs)
correct = {"count_adults": 0, "count_children": 0, "start_date": 0, "nights": 0, "price_per_day": 0}
both = 0

for m, r in zip(predictions, refs):
    a_ok = norm_int(m.get("count_adults")) == r["count_adults"]
    c_ok = norm_int(m.get("count_children")) == r["count_children"]
    correct["count_adults"] += int(a_ok)
    correct["count_children"] += int(c_ok)
    both += int(a_ok and c_ok)
    correct["start_date"] += int(norm_date(m.get("start_date")) == norm_date(r["start_date"]))
    correct["nights"] += int(norm_int(m.get("nights")) == r["nights"])
    correct["price_per_day"] += int(norm_int(m.get("price_per_day")) == r["price_per_day"])

for field, cnt in correct.items():
    print(f"{field}: {cnt}/{total} = {cnt / total:.1%}")

field1 = both / total
field2 = correct["start_date"] / total
field3 = correct["nights"] / total
field4 = correct["price_per_day"] / total
print(f"поле 1 (взрослые+дети вместе): {both}/{total} = {field1:.1%}")
print(f"средняя точность по полям 1-4: {(field1 + field2 + field3 + field4) / 4:.1%}")

### Вывод

- **Поле 1 (взрослые + дети)**: 80.0%. Ошибки — заявка №4 (не извлечено `count_children`) и №8, где «два двухместных номера или 1 на 4 человека» дало 2 взрослых вместо 4; №15 вернула `null`.
- **Поле 2 (start_date)**: 100% — даты извлекаются уверенно.
- **Поле 3 (nights)**: 73.3% — основной источник ошибок: модель считает ночи включительно (например, «с 1.09 по 8.09» → 8 вместо 7).
- **Поле 4 (price_per_day)**: 100%, но поле вырождено — ни в одной заявке цена в сутки не указана числом, поэтому модель корректно возвращает отсутствие значения.

Средняя точность по полям 1-4: **88.3%**. Для повышения точности поля `nights` стоит явно задать правило подсчёта (разница дат, без включения дня выезда).